In [ ]:
"""
Mencari K minimum yang benar-benar valid (deret konvergen secara wajar,
bukan karena pembatalan ekstrem antar suku besar) untuk tiap nilai u.

Kriteria konvergensi:
  1. |term_k| turun di bawah toleransi relatif terhadap suku terbesar
     yang pernah muncul, dan bertahan kecil untuk beberapa suku berturut-turut
     (supaya tidak salah berhenti di titik yang kebetulan kecil sesaat).
  2. Psi(u) yang dihitung dengan K=20 dibandingkan dengan Psi(u) memakai
     K yang jauh lebih besar (K_min + buffer) -> selisihnya jadi ukuran
     "galat truncation" yang sesungguhnya, bukan asumsi.

Presisi (mp.dps) dan jumlah suku dalam deret Mittag-Leffler (n_terms)
disesuaikan otomatis mengikuti besarnya z = (beta2 + lam/c) * u, supaya
tidak berulang mengalami masalah presisi seperti sebelumnya.
"""

from mpmath import mp, mpf, gamma as mgamma, factorial as mfact, exp

# ============================================================
# PARAMETER MODEL (tetap, sesuai dokumen)
# ============================================================
lam   = mpf('0.01657')
alpha = mpf('2.145432')
beta2 = mpf('0.009844')
mu    = mpf('217.95')
E_S   = mpf('242.18')
LAM_MU = lam * mu

TOL_REL       = mpf('1e-12')  # ambang relatif |term_k| terhadap suku terbesar
STABLE_STREAK = 5             # harus tetap di bawah ambang selama n suku berturut
K_HARD_MAX    = 300           # batas keras supaya tidak infinite loop
BUFFER_K      = 30            # tambahan K di atas K_min untuk hitung "psi konvergen"


def adaptive_settings(z):
    """Tentukan presisi (dps) dan jumlah suku dalam (n_terms) mengikuti besar z."""
    z = float(z)
    dps = max(50, int(0.5 * z) + 80)
    n_terms_inner = max(300, int(2.5 * z) + 150)
    return dps, n_terms_inner


def E_deriv(k, z, n_terms):
    """E^(k)_{1, alpha*k+1}(z)"""
    if k == 0:
        return exp(z)
    total = mpf(0)
    for j in range(n_terms):
        total += mfact(j + k) / (mfact(j) * mgamma(j + (alpha + 1) * k + 1)) * z**j
    return total


def compute_terms(u, theta, K_max, n_terms_inner):
    """Hitung daftar term_k (k=0..K_max) dan partial sum berjalan."""
    u = mpf(u)
    theta = mpf(theta)
    c = (1 + theta) * E_S
    coef_theta = lam * (beta2 ** alpha) / c
    ml_base = beta2 + lam / c
    z = ml_base * u

    terms = []
    S = mpf(0)
    max_term_seen = mpf(0)
    for k in range(K_max + 1):
        Ek = E_deriv(k, z, n_terms_inner)
        term_k = ((-1) ** k / mfact(k)) * (coef_theta ** k) * (u ** ((alpha + 1) * k)) * Ek
        S += term_k
        max_term_seen = max(max_term_seen, abs(term_k))
        terms.append((k, term_k, S, max_term_seen))
    return terms, z, c


def find_K_min(terms):
    """Cari K minimum: |term_k| < TOL_REL * max_term_seen, bertahan STABLE_STREAK suku."""
    streak = 0
    for k, term_k, S, max_seen in terms:
        if max_seen > 0 and abs(term_k) < TOL_REL * max_seen:
            streak += 1
            if streak >= STABLE_STREAK:
                return k - STABLE_STREAK + 1
        else:
            streak = 0
    return None  # tidak konvergen dalam K_HARD_MAX suku


def psi_from_terms(terms, K, phi0, decay):
    """Hitung psi(u) memakai partial sum hingga suku ke-K."""
    for k, term_k, S, max_seen in terms:
        if k == K:
            phi_u = phi0 * decay * S
            return float(1 - phi_u)
    return None


def analisis_u(u, theta=0.05, dps_multiplier=1.0):   # tambahkan parameter ini
    theta_mp = mpf(theta)
    c = (1 + theta_mp) * E_S
    phi0 = 1 - LAM_MU / c
    decay = exp(-beta2 * mpf(u))

    ml_base = beta2 + lam / c
    z_est = ml_base * mpf(u)
    dps, n_terms_inner = adaptive_settings(z_est)
    dps = int(dps * dps_multiplier)   # <-- tambahkan baris ini


    mp.dps = dps
    terms, z, c = compute_terms(u, theta, K_HARD_MAX, n_terms_inner)
    K_min = find_K_min(terms)

    if K_min is None:
        return {
            "u": u, "z": float(z), "dps": dps, "n_terms_inner": n_terms_inner,
            "K_min": None, "valid_K20": False,
            "psi_K20": None, "psi_converged": None, "selisih": None,
        }

    K_ref = min(K_min + BUFFER_K, K_HARD_MAX)
    psi_K20 = psi_from_terms(terms, 20, phi0, decay) if len(terms) > 20 else None
    psi_ref = psi_from_terms(terms, K_ref, phi0, decay)

    selisih = abs(psi_K20 - psi_ref) if (psi_K20 is not None and psi_ref is not None) else None
    valid_K20 = (K_min <= 20) and (selisih is not None and selisih < 1e-5)

    return {
        "u": u, "z": float(z), "dps": dps, "n_terms_inner": n_terms_inner,
        "K_min": K_min, "valid_K20": valid_K20,
        "psi_K20": psi_K20, "psi_converged": psi_ref, "selisih": selisih,
    }


if __name__ == "__main__":
    for u_test in [5000, 10000, 14859, 20000, 50000]:
        r = analisis_u(u_test, theta=0.05, dps_multiplier=1.0)
        print(f"u={u_test:>7} -> K_min={r['K_min']}, psi={r['psi_converged']}")
